# Workshop 1 — EDA con NumPy y Pandas: Airbnb Listings

**Integrantes:**
- `Kevin Pabon` — Fases 1 y 2
- `Jose Jimenez` — Fase 3
- `Geronimo Montes Acebedo` — Fase 4 y 5

**Dataset:** Airbnb Listings ([Kaggle](https://www.kaggle.com/datasets/ulrikthygepedersen/airbnb-listings))

**Archivo fuente:** `data/airbnb/Listings.csv`

Este notebook cubre las Fases 1 y 2 del taller (reconocimiento inicial y limpieza). Las Fases 3 a 5 las continúan los compañeros de equipo a partir de la sección "Estado para Fase 3".

## Punto 1 — Descripción del tema del dataset

El dataset contiene **279,712 listados de alojamiento de Airbnb** repartidos en **10 ciudades** de distintos continentes: París, Nueva York, Bangkok, Río de Janeiro, Sídney, Estambul, Roma, Hong Kong, Ciudad de México y Ciudad del Cabo.

Cada fila es un listado individual y describe:
- **El host**: antigüedad en la plataforma, ubicación, tiempo/tasa de respuesta, si es superhost, cantidad de propiedades que administra.
- **La propiedad**: barrio, distrito (solo para Nueva York), ciudad, coordenadas, tipo de propiedad, tipo de habitación, capacidad, número de habitaciones, amenities.
- **Condiciones comerciales**: precio por noche, noches mínimas y máximas de estadía, si permite reserva instantánea.
- **Reputación**: puntajes de reseñas (rating general y 6 sub-categorías: precisión, limpieza, check-in, comunicación, ubicación, relación calidad-precio).

Es un snapshot puntual, no un scraping histórico acumulado: no hay ninguna columna de fecha de scraping/actualización entre las 33 columnas (`host_since` es la fecha de registro del host, no de captura del dato), la metadata de Kaggle marca `expectedUpdateFrequency: never`, y no hay `listing_id` repetidos que indiquen el mismo listado capturado en momentos distintos.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = "data/airbnb/Listings.csv"

# El CSV es UTF-8 valido en casi la totalidad de sus filas, pero tiene un puñado
# de bytes corruptos de origen (ej. un caracter mal guardado en un nombre de barrio).
# encoding_errors="replace" sustituye SOLO esos bytes puntuales por el caracter de
# reemplazo U+FFFD, sin degradar el resto del archivo (a diferencia de forzar
# encoding="latin-1" para todo el archivo, que rompe el texto UTF-8 que si es valido).
df = pd.read_csv(DATA_PATH, encoding="utf-8", encoding_errors="replace", low_memory=False)

print("Shape inicial:", df.shape)


Shape inicial: (279712, 33)


## Punto 2 — head() y tail()

Primeras y últimas 5 filas del dataset, tal como vienen del CSV original (sin limpieza todavía).


In [2]:
print("HEAD(5):")
display(df.head(5))

print("\nTAIL(5):")
display(df.tail(5))


HEAD(5):


,listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood,district,city,latitude,longitude,property_type,room_type,accommodates,bedrooms,amenities,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
0,281420,"Beautiful Flat in le Village Montmartre, Paris",1466919,2011-12-03,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,f,Buttes-Montmartre,NaN,Paris,48.88668,2.33343,Entire apartment,Entire place,2,1.0,"[""Heating"", ""Kitchen"", ""Washer"", ""Wifi"", ""Long...",53,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
1,3705183,39 mÂ² Paris (Sacre CÅ“ur),10328771,2013-11-29,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,t,Buttes-Montmartre,NaN,Paris,48.88617,2.34515,Entire apartment,Entire place,2,1.0,"[""Shampoo"", ""Heating"", ""Kitchen"", ""Essentials""...",120,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
2,4082273,"Lovely apartment with Terrace, 60m2",19252768,2014-07-31,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,f,Elysee,NaN,Paris,48.88112,2.31712,Entire apartment,Entire place,2,1.0,"[""Heating"", ""TV"", ""Kitchen"", ""Washer"", ""Wifi"",...",89,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
3,4797344,Cosy studio (close to Eiffel tower),10668311,2013-12-17,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,t,Vaugirard,NaN,Paris,48.84571,2.30584,Entire apartment,Entire place,2,1.0,"[""Heating"", ""TV"", ""Kitchen"", ""Wifi"", ""Long ter...",58,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
4,4823489,Close to Eiffel Tower - Beautiful flat : 2 rooms,24837558,2014-12-14,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,f,Passy,NaN,Paris,48.85500,2.26979,Entire apartment,Entire place,2,1.0,"[""Heating"", ""TV"", ""Kitchen"", ""Essentials"", ""Ha...",60,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f



TAIL(5):


,listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood,district,city,latitude,longitude,property_type,room_type,accommodates,bedrooms,amenities,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
279707,38338635,Appartement T2 neuf prÃ¨s du tram T3a Porte Didot,31161181,2015-04-13,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,t,Observatoire,NaN,Paris,48.82701,2.31419,Entire apartment,Entire place,2,1.0,"[""Iron"", ""Heating"", ""Washer"", ""Dedicated works...",120,1,7,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
279708,38538692,Cozy Studio in Montmartre,10294858,2013-11-27,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,t,Buttes-Montmartre,NaN,Paris,48.89309,2.33206,Entire apartment,Entire place,2,1.0,"[""Shampoo"", ""Iron"", ""Heating"", ""Washer"", ""Hair...",60,7,15,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
279709,38683356,Nice and cosy mini-appartement in Paris,2238502,2012-04-27,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,t,Buttes-Montmartre,NaN,Paris,48.88699,2.34920,Entire apartment,Entire place,2,1.0,"[""Paid parking off premises"", ""Shampoo"", ""Firs...",50,6,30,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
279710,39659000,Charming apartment near Rue Saint Maur / Oberk...,38633695,2015-07-16,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,t,Popincourt,NaN,Paris,48.86687,2.38123,Entire apartment,Entire place,2,1.0,"[""TV"", ""Iron"", ""Kitchen"", ""Hangers"", ""Smoke al...",105,3,18,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
279711,40219504,Cosy apartment with view on Canal St Martin,6955618,2013-06-17,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,t,t,Enclos-St-Laurent,NaN,Paris,48.87217,2.36320,Entire apartment,Entire place,2,1.0,"[""Shower gel"", ""Shampoo"", ""Iron"", ""Heating"", ""...",70,2,4,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f


## Punto 3 — .shape y diccionario de datos

El dataset tiene **279,712 filas × 33 columnas**. Diccionario de datos completo:

| Columna | Tipo esperado | Significado |
|---|---|---|
| listing_id | numérica (entero, ID) | Identificador único del listado |
| name | categórica (texto libre) | Título/nombre del anuncio |
| host_id | numérica (entero, ID) | Identificador único del host |
| host_since | fecha | Fecha en que el host se registró en Airbnb |
| host_location | categórica (texto libre) | Ciudad/país declarado por el host |
| host_response_time | categórica | Rapidez típica de respuesta del host |
| host_response_rate | numérica (%) | % de mensajes que el host responde |
| host_acceptance_rate | numérica (%) | % de solicitudes de reserva que el host acepta |
| host_is_superhost | categórica (booleana f/t) | Si el host tiene estatus "superhost" |
| host_total_listings_count | numérica (entero) | Cantidad total de propiedades que administra el host |
| host_has_profile_pic | categórica (booleana f/t) | Si el host tiene foto de perfil |
| host_identity_verified | categórica (booleana f/t) | Si Airbnb verificó la identidad del host |
| neighbourhood | categórica | Barrio dentro de la ciudad |
| district | categórica | Distrito (solo poblado para Nueva York) |
| city | categórica | Ciudad del listado (10 valores posibles) |
| latitude | numérica (continua) | Latitud geográfica |
| longitude | numérica (continua) | Longitud geográfica |
| property_type | categórica | Tipo de propiedad (apartamento, casa, loft, etc.) |
| room_type | categórica | Tipo de habitación ofrecida (lugar completo, privada, compartida, hotel) |
| accommodates | numérica (entero) | Capacidad máxima de huéspedes |
| bedrooms | numérica (entero, con nulos) | Número de habitaciones |
| amenities | categórica (texto tipo lista) | Lista de comodidades ofrecidas, en formato string JSON |
| price | numérica (entero) | Precio por noche, **en moneda local de cada ciudad** |
| minimum_nights | numérica (entero) | Noches mínimas exigidas por reserva |
| maximum_nights | numérica (entero) | Noches máximas permitidas por reserva |
| review_scores_rating | numérica (0-100) | Puntaje general de reseñas |
| review_scores_accuracy | numérica (0-10) | Puntaje de precisión de la descripción |
| review_scores_cleanliness | numérica (0-10) | Puntaje de limpieza |
| review_scores_checkin | numérica (0-10) | Puntaje del proceso de check-in |
| review_scores_communication | numérica (0-10) | Puntaje de comunicación con el host |
| review_scores_location | numérica (0-10) | Puntaje de ubicación |
| review_scores_value | numérica (0-10) | Puntaje de relación calidad-precio |
| instant_bookable | categórica (booleana f/t) | Si el listado permite reserva instantánea sin aprobación |

**Nota sobre las escalas de reseñas:** `review_scores_rating` usa escala **0-100**, mientras que las 6 sub-métricas (`accuracy`, `cleanliness`, `checkin`, `communication`, `location`, `value`) usan escala **0-10**. Verificado con `.describe()`: el rating general va de 20 a 100. No son directamente comparables entre sí sin normalizar.


In [3]:
print("Shape:", df.shape)
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("\nNombres de columnas:")
print(list(df.columns))


Shape: (279712, 33)
Filas: 279712
Columnas: 33

Nombres de columnas:
['listing_id', 'name', 'host_id', 'host_since', 'host_location', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_total_listings_count', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'district', 'city', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bedrooms', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable']


## Punto 4 — .info() y .dtypes

Clasificación de cada columna:

**Numéricas:** `listing_id`, `host_id`, `host_response_rate`, `host_acceptance_rate`, `host_total_listings_count`, `latitude`, `longitude`, `accommodates`, `bedrooms`, `price`, `minimum_nights`, `maximum_nights`, `review_scores_rating`, `review_scores_accuracy`, `review_scores_cleanliness`, `review_scores_checkin`, `review_scores_communication`, `review_scores_location`, `review_scores_value`.

**Categóricas:** `name`, `host_location`, `host_response_time`, `host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `neighbourhood`, `district`, `city`, `property_type`, `room_type`, `amenities`, `instant_bookable`.

**Fecha:** `host_since` — **debería ser tipo fecha, pero pandas la carga como `object` (string)** porque el CSV no trae un tipo de dato nativo. La conversión real con `pd.to_datetime` se hace en el Punto 10 (Fase 2).


In [4]:
df.info()
print("\nDtypes:")
print(df.dtypes)


<class 'pandas.DataFrame'>
RangeIndex: 279712 entries, 0 to 279711
Data columns (total 33 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   listing_id                   279712 non-null  int64  
 1   name                         279537 non-null  str    
 2   host_id                      279712 non-null  int64  
 3   host_since                   279547 non-null  str    
 4   host_location                278872 non-null  str    
 5   host_response_time           150930 non-null  str    
 6   host_response_rate           150930 non-null  float64
 7   host_acceptance_rate         166625 non-null  float64
 8   host_is_superhost            279547 non-null  str    
 9   host_total_listings_count    279547 non-null  float64
 10  host_has_profile_pic         279547 non-null  str    
 11  host_identity_verified       279547 non-null  str    
 12  neighbourhood                279712 non-null  str    
 13  district  

## Punto 5 — .describe() e interpretación

Interpretación de al menos 3 estadísticos en términos del dominio:

- **`price`**: la media global mezcla 10 monedas locales distintas (ver Punto 1 y verificación previa: medianas van de 65 en Roma a 1100 en Bangkok). **La media/std de `price` calculada sobre todo el dataset NO es interpretable como "precio real"** — no representa ni un promedio en una sola moneda ni un promedio convertido. Solo es útil por ciudad (ver Punto 6).
- **`accommodates`**: la mediana es 2 huéspedes, coherente con que la mayoría de los listados son apartamentos completos o habitaciones privadas pensadas para parejas o viajeros individuales, no para grupos grandes.
- **`review_scores_rating`** (escala **0-100**, a diferencia de las 6 sub-métricas que van de 0 a 10): la distribución está fuertemente sesgada hacia lo alto — media 93.4, mediana 96 y Q1 91, con mínimo 20 y máximo 100. Que el 75% de los listados califique por encima de 91 significa que una puntuación "buena" no distingue casi nada: lo informativo es caer por debajo de ~90, que ya ubica al alojamiento en el cuartil inferior. Es el sesgo de reseñas positivas típico de las plataformas de hospedaje, donde el huésped insatisfecho tiende a no reseñar o a cancelar antes de completar la estadía.
- **`maximum_nights`**: la media (~27,558) está muy por encima de la mediana (1125) por la presencia de placeholders extremos como `2147483647` (`INT32_MAX`), que representan "sin límite" y no una noche máxima real — se detalla en el Punto 11.

Los placeholders de `maximum_nights` se dejan crudos aquí a propósito: este punto es el reconocimiento inicial del dataset "tal cual llega", antes de cualquier limpieza. El tratamiento real de esos placeholders se hace en el Punto 11 (Fase 2).

In [5]:
display(df.describe())


,listing_id,host_id,host_response_rate,host_acceptance_rate,host_total_listings_count,latitude,longitude,accommodates,bedrooms,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
count,2.797120e+05,2.797120e+05,150930.000000,166625.000000,279547.000000,279712.000000,279712.000000,279712.000000,250277.000000,279712.000000,279712.000000,2.797120e+05,188307.000000,187999.000000,188047.000000,187941.000000,188025.000000,187937.000000,187927.000000
mean,2.638196e+07,1.081658e+08,0.865939,0.827168,24.581612,18.761862,12.595075,3.288736,1.515509,608.792737,8.050967,2.755860e+04,93.405195,9.565476,9.312869,9.701534,9.698593,9.633994,9.335364
std,1.442576e+07,1.108570e+08,0.283744,0.289202,284.041143,32.560343,73.081309,2.133379,1.153080,3441.826611,31.518946,7.282875e+06,10.070437,0.990878,1.146072,0.867434,0.886884,0.833234,1.042625
min,2.577000e+03,1.822000e+03,0.000000,0.000000,0.000000,-34.264400,-99.339630,0.000000,1.000000,0.000000,1.000000,1.000000e+00,20.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
25%,1.384462e+07,1.720656e+07,0.900000,0.780000,1.000000,-22.964390,-43.198040,2.000000,1.000000,75.000000,1.000000,4.500000e+01,91.000000,9.000000,9.000000,10.000000,10.000000,9.000000,9.000000
50%,2.767098e+07,5.826911e+07,1.000000,0.980000,1.000000,40.710785,2.382780,2.000000,1.000000,150.000000,2.000000,1.125000e+03,96.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
75%,3.978485e+07,1.832853e+08,1.000000,1.000000,4.000000,41.908610,28.986730,4.000000,2.000000,474.000000,5.000000,1.125000e+03,100.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
max,4.834353e+07,3.901874e+08,1.000000,1.000000,7235.000000,48.904910,151.339810,16.000000,50.000000,625216.000000,9999.000000,2.147484e+09,100.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000


## Punto 6 — Conversión a NumPy: media, mediana y std de `price`

Como se confirmó que `price` está en **moneda local por ciudad** (no hay una moneda común ni tasas de cambio en el dataset), calcular media/mediana/std sobre las 279,712 filas mezclaría monedas y el resultado no tendría significado económico.

Por eso se **filtra a una sola ciudad** antes de convertir a NumPy: se eligió **París**, por ser la ciudad con más registros (64,690 listados).


In [6]:
city_top = df["city"].value_counts().idxmax()
print("Ciudad con mas registros:", city_top, "->", df["city"].value_counts().max(), "listados")

price_city_np = df.loc[df["city"] == city_top, "price"].to_numpy()

media_np = np.mean(price_city_np)
mediana_np = np.median(price_city_np)
std_np = np.std(price_city_np)          # ddof=0 (poblacional), default de NumPy
std_np_ddof1 = np.std(price_city_np, ddof=1)  # ddof=1 (muestral), para comparar con pandas

print(f"\nNumPy (price en {city_top}, n={len(price_city_np)}):")
print(f"  media   = {media_np:.2f}")
print(f"  mediana = {mediana_np:.2f}")
print(f"  std (ddof=0, poblacional) = {std_np:.2f}")
print(f"  std (ddof=1, muestral)    = {std_np_ddof1:.2f}")

desc_city = df.loc[df["city"] == city_top, "price"].describe()
print(f"\n.describe() de price en {city_top}:")
print(desc_city)


Ciudad con mas registros: Paris -> 64690 listados

NumPy (price en Paris, n=64690):
  media   = 113.10
  mediana = 80.00
  std (ddof=0, poblacional) = 214.43
  std (ddof=1, muestral)    = 214.43

.describe() de price en Paris:
count    64690.000000
mean       113.096445
std        214.433668
min          0.000000
25%         59.000000
50%         80.000000
75%        120.000000
max      12000.000000
Name: price, dtype: float64


**Comparación:** la media y la mediana calculadas a mano con NumPy coinciden exactamente con las de `.describe()` (ambas parten de los mismos datos y fórmulas). La única diferencia aparece en la **desviación estándar**: `np.std()` usa por defecto `ddof=0` (divide entre N, desviación poblacional), mientras que `.describe()` de pandas usa `ddof=1` (divide entre N-1, desviación muestral) — por eso se calculó también `np.std(..., ddof=1)`, que sí coincide con pandas. Con un tamaño de muestra tan grande (n=64,690) la diferencia entre ambas es mínima en términos relativos, pero es importante saber cuál usa cada herramienta por defecto para no comparar resultados de forma incorrecta.


## Punto 7 — Nulos por columna (conteo y %)

Conteo y porcentaje de valores nulos sobre el total de filas (279,712), para las columnas que tienen al menos un nulo.


In [7]:
nulls = df.isnull().sum()
pct = (nulls / len(df) * 100).round(2)
null_report = pd.DataFrame({"nulos": nulls, "pct_%": pct})
null_report = null_report[null_report["nulos"] > 0].sort_values("nulos", ascending=False)
display(null_report)

print("\nColumnas sin nulos:")
print(list(df.columns[df.isnull().sum() == 0]))


,nulos,pct_%
district,242700,86.77
host_response_time,128782,46.04
host_response_rate,128782,46.04
host_acceptance_rate,113087,40.43
review_scores_value,91785,32.81
review_scores_location,91775,32.81
review_scores_checkin,91771,32.81
review_scores_accuracy,91713,32.79
review_scores_communication,91687,32.78
review_scores_cleanliness,91665,32.77



Columnas sin nulos:


['listing_id', 'host_id', 'neighbourhood', 'city', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'instant_bookable']


## Punto 8 — Estrategia de nulos justificada por columna

**Regla del taller: no se elimina ninguna columna completa, ni siquiera `district` con 86.77% de nulos.**

Se clasifica cada columna con nulos en **estructural** (el nulo significa "no aplica" / "no hay dato por diseño") o **aleatorio** (falta de captura, se puede imputar razonablemente).

### Estructurales — NO se imputan con media/mediana/moda, se deja NaN o constante + flag booleana

| Columna | Estrategia | Justificación |
|---|---|---|
| `district` | Mantener NaN | Solo existe para Nueva York (por diseño del scraping); para las otras 9 ciudades no aplica el concepto de "distrito". |
| `host_response_time` | Constante `"Sin datos"` + flag | Nulo cuando el host nunca ha recibido/respondido mensajes — no es un dato faltante por error, es ausencia de historial. |
| `host_response_rate` | Mantener NaN + flag | Mismo caso que `host_response_time` (nulos en las mismas 128,782 filas exactas, verificado). |
| `review_scores_rating` y las 6 sub-métricas | Mantener NaN + flag `tiene_resenas` | Nulo cuando el listado no tiene reseñas todavía. Imputar con la media escondería que el listado es nuevo/sin evaluar. **Nota de granularidad:** `review_scores_rating` es nulo en 91,405 filas; las 7 columnas nulas simultáneamente en 91,377 filas; al menos una nula en 91,881 filas. La diferencia (91,881 − 91,377 = 504 filas con nulos parciales, de las cuales 476 corresponden al patrón "rating con valor pero alguna sub-métrica nula") indica que el patrón es *casi* estructural pero no perfecto — se documenta explícitamente en vez de asumir uniformidad. |

### Aleatorios — se imputan o se elimina la fila, justificado

| Columna | % nulos | Estrategia | Justificación |
|---|---|---|---|
| `bedrooms` | 10.52% | Imputar con la **mediana** | Es numérica con distribución sesgada (pocas propiedades con muchas habitaciones); la mediana es más robusta que la media ante ese sesgo. |
| `host_location` | 0.30% (840 filas) | Mantener NaN — **no se elimina ninguna fila por esta columna** | Dato declarado libremente por el host, no crítico para el análisis y muy bajo %. Es texto libre, así que imputarlo sería inventar una ubicación: se sale del alcance de Fase 2. |
| `name` | 0.06% | Mantener NaN | Campo de texto libre, imputar un nombre inventado no aporta valor analítico. |
| `host_since`, `host_identity_verified`, `host_has_profile_pic`, `host_total_listings_count`, `host_is_superhost` | 0.06% cada una | Eliminar esas filas | Ver justificación abajo: son exactamente las mismas 165 filas. |

### Por qué se eliminan las 165 filas de metadata de host

Los nulos de `host_since`, `host_is_superhost`, `host_has_profile_pic` y `host_identity_verified` **están en las mismas 165 filas**: cada columna tiene 165 nulos por separado, y las 4 nulas *simultáneamente* también dan 165 (se verifica en la celda de abajo). Es un bloque de metadata de host ausente en conjunto, probablemente cuentas eliminadas de la plataforma.

Por eso el `dropna` sobre ese subconjunto elimina **exactamente 165 filas, no más**: no se van sumando filas distintas por cada columna. La eliminación queda justificada porque una fila sin ninguna metadata de host aporta poco al análisis y el volumen es despreciable (0.06% del dataset).

### `host_location` NO pertenece a ese bloque

`host_location` tiene **840 nulos**, no 165 — es una columna aparte, con nulos aleatorios propios. Las ~675 filas cuyo `host_location` es nulo pero que sí tienen metadata de host completa **sobreviven a la limpieza y conservan su NaN**. `host_location` nunca entra al `dropna`.


In [8]:
df_clean = df.copy()

# --- Estructurales: flags + constante/NaN, SIN imputar con media/mediana/moda ---
df_clean["tiene_resenas"] = df_clean["review_scores_rating"].notna()
df_clean["tiene_respuesta_host"] = df_clean["host_response_time"].notna()

df_clean["host_response_time"] = df_clean["host_response_time"].fillna("Sin datos")
# host_response_rate, review_scores_* y district se dejan en NaN (estructural, no se imputan)

# --- Aleatorios ---
mediana_bedrooms = df_clean["bedrooms"].median()
df_clean["bedrooms"] = df_clean["bedrooms"].fillna(mediana_bedrooms)
print(f"bedrooms: nulos imputados con mediana = {mediana_bedrooms}")

# PRUEBA: los nulos de metadata de host estan en las MISMAS filas
cols_bloque_host = ["host_since", "host_is_superhost", "host_has_profile_pic",
                    "host_identity_verified"]
print("\nNulos por columna del bloque de metadata de host:")
print(df[cols_bloque_host].isnull().sum())
n_bloque = df[cols_bloque_host].isnull().all(axis=1).sum()
print(f"\nFilas con las 4 columnas nulas SIMULTANEAMENTE: {n_bloque}")
print("-> Coincide con los 165 nulos de cada columna: son las mismas 165 filas.")

# host_location NO pertenece a ese bloque: tiene sus propios nulos, mas numerosos
print(f"\nhost_location nulos: {df['host_location'].isnull().sum()} (NO son las mismas 165 filas)")
solo_host_location = (df["host_location"].isnull() & df[cols_bloque_host].notna().all(axis=1)).sum()
print(f"Filas con host_location nulo pero metadata de host completa: {solo_host_location}")
print("-> Esas filas sobreviven a la limpieza y conservan su NaN en host_location.")

filas_antes = len(df_clean)
cols_host_meta = ["host_since", "host_identity_verified", "host_has_profile_pic",
                   "host_total_listings_count", "host_is_superhost"]
df_clean = df_clean.dropna(subset=cols_host_meta)
print(f"\nFilas eliminadas por metadata de host incompleta: {filas_antes - len(df_clean)}")

# host_location y name: se dejan como estan (NaN), no se imputan ni se eliminan filas
print(f"host_location nulos que sobreviven en df_clean: {df_clean['host_location'].isnull().sum()}")

print("\nVerificacion post-limpieza (isnull().sum()):")
print(df_clean.isnull().sum())


bedrooms: nulos imputados con mediana = 1.0

Nulos por columna del bloque de metadata de host:
host_since                165
host_is_superhost         165
host_has_profile_pic      165
host_identity_verified    165
dtype: int64

Filas con las 4 columnas nulas SIMULTANEAMENTE: 165
-> Coincide con los 165 nulos de cada columna: son las mismas 165 filas.

host_location nulos: 840 (NO son las mismas 165 filas)


Filas con host_location nulo pero metadata de host completa: 675
-> Esas filas sobreviven a la limpieza y conservan su NaN en host_location.

Filas eliminadas por metadata de host incompleta: 165
host_location nulos que sobreviven en df_clean: 675

Verificacion post-limpieza (isnull().sum()):


listing_id                          0
name                              174
host_id                             0
host_since                          0
host_location                     675
host_response_time                  0
host_response_rate             128617
host_acceptance_rate           112922
host_is_superhost                   0
host_total_listings_count           0
host_has_profile_pic                0
host_identity_verified              0
neighbourhood                       0
district                       242553
city                                0
latitude                            0
longitude                           0
property_type                       0
room_type                           0
accommodates                        0
bedrooms                            0
amenities                           0
price                               0
minimum_nights                      0
maximum_nights                      0
review_scores_rating            91348
review_score

## Punto 9 — Duplicados

Se verifica duplicados exactos de fila y duplicados lógicos por `listing_id` (que debería ser único).


In [9]:
dup_exact = df_clean.duplicated().sum()
print("Duplicados exactos de fila:", dup_exact)

if dup_exact > 0:
    df_clean = df_clean.drop_duplicates()
    print("Filas eliminadas por duplicado exacto:", dup_exact)
else:
    print("No se eliminó ninguna fila: 0 duplicados exactos encontrados.")

dup_ids = df_clean["listing_id"].duplicated().sum()
print("\nDuplicados logicos por listing_id:", dup_ids)
print("listing_id unicos:", df_clean['listing_id'].nunique(), "vs filas totales:", len(df_clean))
print("-> Cada fila corresponde a un listing_id unico, no hay duplicados logicos.")


Duplicados exactos de fila: 0
No se eliminó ninguna fila: 0 duplicados exactos encontrados.

Duplicados logicos por listing_id: 0
listing_id unicos: 279547 vs filas totales: 279547
-> Cada fila corresponde a un listing_id unico, no hay duplicados logicos.


## Punto 10 — Inconsistencias de formato en columnas de texto

Primero se revisan con `.unique()` las columnas categóricas "cerradas" (pocas categorías posibles): `room_type` y `city`. Ambas ya salieron consistentes en el reconocimiento inicial (Paso 0) — sin mayúsculas mezcladas, espacios extra ni categorías equivalentes escritas distinto — así que aquí se confirma con `.unique()` y no requieren corrección.

Luego se corrigen tres tipos de inconsistencia:

1. **Encoding (doble codificación / "mojibake"):** buena parte del texto del CSV se guardó doblemente codificado, así que llega como `"PÃ¨re Lachaise"` en vez de `"Père Lachaise"` o `"39 mÂ² ... CÅ“ur"` en vez de `"39 m² ... Cœur"`. Se corrige deshaciendo esa capa (`.encode("cp1252").decode("utf-8")`), aplicada **solo cuando el resultado mejora de verdad**: hay filas del dataset cuyo texto ya está bien y aplicarles la conversión a ciegas las rompería, así que la función descarta el cambio si introduce caracteres de reemplazo o no reduce los marcadores. Se usa `cp1252` y no `latin-1` porque el texto contiene comillas tipográficas (`“`, U+201C) y otros caracteres que no existen en latin-1 y harían fallar la conversión.

**Resultado: la corrección recupera ~98.6% del mojibake de `name` (25,697 → 449 filas residuales).** Ese residuo se desglosa abajo, y es importante no confundirlo con un segundo problema distinto (los bytes corruptos de origen).

#### Las 449 filas residuales: dos casos, ninguno se corrige

- **360 filas — mojibake real que la función no alcanza.** Su mojibake viene mezclado con **emojis y símbolos** (☀ ❤ ★, kanji) cuya representación rota cae fuera del rango de `cp1252`, así que `.encode("cp1252")` lanza `UnicodeEncodeError` y la función las deja intactas **por diseño**: prefiere no tocar antes que corromper. **Decisión: se reportan, no se corrigen.** `name` es texto libre que no se usa en `groupby` ni en las visualizaciones de las Fases 3-4, y una corrección robusta de mojibake combinado con emojis excede el alcance de la Fase 2.
- **89 filas — no son mojibake: es texto legítimo en portugués.** Son títulos de listados de Río de Janeiro donde `Ã` es la letra real del idioma: *coraçÃO*, *localizaçÃO*, *MaracanÃ*, *espigÃO*. La guarda "solo corregir si mejora" **hizo lo correcto al no tocarlas**; aparecen en el conteo solo porque el detector busca el carácter `Ã` sin distinguir el contexto en que aparece. Corregirlas sería el error.

#### Aparte: 5 filas con byte corrupto de origen

Independientemente de lo anterior, el archivo trae unos pocos **bytes corruptos desde la fuente** (un `0x81` en "Álvaro Obregón"). Se cargó con `encoding_errors="replace"` en el Punto 1, así que quedan marcados con `�`. **Estas 5 filas NO forman parte de las 449**: su texto no contiene ningún marcador `Ã`/`Â`/`Å`, por lo que nunca entraron al conteo de mojibake. Son dos grupos **disjuntos** (la celda verifica que la intersección es 0), con causas distintas: las 449 son un problema de *codificación* reversible en parte, y estas 5 son *pérdida de información* en el archivo original — no se pueden reconstruir sin inventar el dato, así que solo se reportan.
2. **Booleanos como texto (`'f'`/`'t'`)** en `host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `instant_bookable` → se convierten a `True`/`False` reales.
3. **`host_since` como string** → se convierte a fecha real con `pd.to_datetime(errors='coerce')`. **Nota:** los 165 valores nulos originales de `host_since` (ver Punto 7) ya se eliminaron junto con el resto de metadata de host incompleta en el Punto 8 — por eso esta conversión da 0 NaT: no es que `host_since` nunca haya tenido problemas, es que ya se resolvieron dos pasos atrás.

`host_location` y `amenities` quedan **solo reportadas** (con `.value_counts().head(15)`, sin `.unique()` completo por tener miles de valores distintos) — normalizarlas se sale del alcance de esta fase y queda para quien continúe con el análisis geográfico/de amenities.


In [10]:
# 0. Columnas categoricas cerradas: verificar con .unique() que ya son consistentes
print("room_type.unique():", df_clean["room_type"].unique())
print("\ncity.unique():", df_clean["city"].unique())
print("\n-> Ambas ya son consistentes: sin mayusculas mezcladas, espacios extra ni")
print("   categorias equivalentes escritas distinto. No requieren correccion.")

# 1. Encoding: deshacer la doble codificacion (mojibake)
MARCADORES = ("\u00c2", "\u00c3", "\u00c5")   # A-circunflejo, A-tilde, A-anillo: senales tipicas de mojibake

def _n_marcadores(s):
    return sum(s.count(m) for m in MARCADORES)

def fix_mojibake(text, max_pasadas=3):
    """Deshace capas de doble codificacion mientras el resultado mejore de forma
    estricta. Si la conversion introduce caracteres de reemplazo o no reduce los
    marcadores, se descarta y se conserva el texto original."""
    if not isinstance(text, str):
        return text
    actual = text
    for _ in range(max_pasadas):
        if not any(m in actual for m in MARCADORES):
            break
        candidato = None
        for enc in ("cp1252", "latin-1"):
            try:
                candidato = actual.encode(enc).decode("utf-8")
                break
            except (UnicodeEncodeError, UnicodeDecodeError):
                continue
        if candidato is None:
            break
        introduce_basura = "\ufffd" in candidato and "\ufffd" not in actual
        if introduce_basura or _n_marcadores(candidato) >= _n_marcadores(actual):
            break
        actual = candidato
    return actual

def _filas_con_mojibake(serie):
    """Cuenta filas DISTINTAS con al menos un marcador. Se combinan las mascaras con
    OR en vez de sumar un contains por marcador: una fila con 'Ã' y 'Â' a la vez
    debe contar una sola vez, no dos."""
    s = serie.astype(str)
    mascara = s.str.contains(MARCADORES[0], na=False)
    for m in MARCADORES[1:]:
        mascara = mascara | s.str.contains(m, na=False)
    return int(mascara.sum())

cols_texto = ["name", "host_location", "neighbourhood"]
print("\nMojibake por columna (filas afectadas) ANTES:")
for c in cols_texto:
    print(f"  {c}: {_filas_con_mojibake(df_clean[c])}")

ejemplos_antes = df_clean.loc[
    df_clean["name"].astype(str).str.contains("\u00c3", na=False), "name"
].head(3).tolist()

for c in cols_texto:
    df_clean[c] = df_clean[c].map(fix_mojibake)

print("\nMojibake por columna (filas afectadas) DESPUES:")
for c in cols_texto:
    print(f"  {c}: {_filas_con_mojibake(df_clean[c])}")

print("\nEjemplos ANTES :", ejemplos_antes)
print("Ejemplos DESPUES:", [fix_mojibake(x) for x in ejemplos_antes])

# --- Desglose de las filas residuales de 'name' ---
# Un marcador suelto NO implica mojibake: en portugues 'Ã' es letra real
# (coracao, localizacao, Maracana). Se separa mojibake real de texto legitimo.
import re
PATRON_SOSPECHOSO = re.compile(
    "[\u00c2\u00c3\u00c5][^\\sA-Za-z]"   # marcador seguido de algo que no es letra/espacio
    "|\u00e2|\u00ef\u00bf|\u00ef\u00b8|\u00e3\u201a"  # restos de emoji/simbolo mal decodificado
)

nombres = df_clean["name"].map(lambda v: v if isinstance(v, str) else "")
mask_resid = nombres.map(lambda v: any(m in v for m in MARCADORES))
residuales = nombres[mask_resid]

es_mojibake_real = residuales.map(lambda v: bool(PATRON_SOSPECHOSO.search(v)))
n_real = int(es_mojibake_real.sum())
n_legitimo = int((~es_mojibake_real).sum())

print(f"\nDesglose de las {len(residuales)} filas residuales de 'name':")
print(f"  - mojibake real NO corregido (simbolos fuera de cp1252): {n_real}")
print(f"  - texto legitimo en portugues (falso positivo del detector): {n_legitimo}")
print("\n  Ejemplos de mojibake real no corregido:")
for x in residuales[es_mojibake_real].head(4).tolist():
    print("   ", x)
print("\n  Ejemplos de texto legitimo (Ã es la letra real, NO se debe tocar):")
for x in residuales[~es_mojibake_real].head(4).tolist():
    print("   ", x)

# --- Conjunto APARTE: bytes corruptos de origen (no solapa con los residuales) ---
mask_fffd = nombres.str.contains("\ufffd", na=False)
n_irrecuperable = int(mask_fffd.sum())
solapan = int((mask_resid & mask_fffd).sum())
print(f"\nGrupo APARTE - filas con caracter irrecuperable (U+FFFD, byte 0x81 corrupto de origen): {n_irrecuperable}")
print(f"Filas que estan en AMBOS grupos: {solapan} -> son conjuntos disjuntos.")
print("Ejemplos:", nombres[mask_fffd].head(3).tolist())

# 2. Booleanos f/t -> True/False
bool_map = {"f": False, "t": True}
bool_cols = ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "instant_bookable"]
for c in bool_cols:
    df_clean[c] = df_clean[c].map(bool_map)
print("\nDtypes tras conversion booleana:")
print(df_clean[bool_cols].dtypes)

# 3. host_since -> fecha real
antes_na = df_clean["host_since"].isnull().sum()
df_clean["host_since"] = pd.to_datetime(df_clean["host_since"], errors="coerce")
print(f"\nhost_since convertido a datetime. Nulos/NaT: {df_clean['host_since'].isnull().sum()} (antes de esta conversion: {antes_na}, ya sin nulos por la limpieza del Punto 8)")

# 4. Reporte de host_location y amenities SIN normalizar (fuera de alcance)
print("\nhost_location - top 15 valores mas frecuentes (inconsistencias: mezcla 'Ciudad, Region, Pais' con codigos ISO):")
print(df_clean["host_location"].value_counts().head(15))

print("\namenities - top 15 combinaciones mas frecuentes (formato string tipo lista, sin normalizar):")
print(df_clean["amenities"].value_counts().head(15))


room_type.unique(): <StringArray>
['Entire place', 'Private room', 'Hotel room', 'Shared room']
Length: 4, dtype: str

city.unique(): <StringArray>
['Paris', 'New York', 'Bangkok', 'Rio de Janeiro', 'Sydney', 'Istanbul', 'Rome', 'Hong Kong', 'Mexico City', 'Cape Town']
Length: 10, dtype: str

-> Ambas ya son consistentes: sin mayusculas mezcladas, espacios extra ni
   categorias equivalentes escritas distinto. No requieren correccion.

Mojibake por columna (filas afectadas) ANTES:
  name: 24206
  host_location: 2788


  neighbourhood: 0



Mojibake por columna (filas afectadas) DESPUES:
  name: 449
  host_location: 4
  neighbourhood: 0

Ejemplos ANTES : ['57sqm btw. Bastille & PÃ¨re Lachaise', 'Beau 2piÃ¨ces prÃ¨s Butte aux cailles', 'Grand 2 piÃ¨ces de 60 m2 cosy au canal Saint Martin']
Ejemplos DESPUES: ['57sqm btw. Bastille & Père Lachaise', 'Beau 2pièces près Butte aux cailles', 'Grand 2 pièces de 60 m2 cosy au canal Saint Martin']



Desglose de las 449 filas residuales de 'name':
  - mojibake real NO corregido (simbolos fuera de cp1252): 356
  - texto legitimo en portugues (falso positivo del detector): 93

  Ejemplos de mojibake real no corregido:
    ã‚ˆã†ã“ã-GrÃ¼ezi-Bienvenue-Welcome Paris Le Marais
    Romantique Appartement Matignon Ã‰lysees â¤
    â˜€ï¸ Magnifique studio Paris 16Ã¨me
    â¤ï¸  AMAZING - 6 guests near to Champs-Ã‰lyseesâ¤ï¸

  Ejemplos de texto legitimo (Ã es la letra real, NO se debe tocar):
    Alugo Ap para CARNAVAL !!!! Ótima LOCALIZAÇÃO !
    APTO EM EXCELENTE LOCALIZAÇÃO
    CHARMOSO NO CORAÇÃO DO PROJAC
    ÓTIMA LOCALIZAÇÃO, AP JACAREPAGUa- RIO DE JANEIRO

Grupo APARTE - filas con caracter irrecuperable (U+FFFD, byte 0x81 corrupto de origen): 5
Filas que estan en AMBOS grupos: 0 -> son conjuntos disjuntos.
Ejemplos: ['Newly Renovated apt in front of A�lvaro Obregon!!', 'Departamento completo en A�lvaro Obregon.', 'A�lvaro Obregon/ Roma cozy Great location 3']

Dtypes tras c

## Punto 11 — Outliers por IQR (a mano) y basura no estadística

**No se elimina nada en este punto**, solo se reporta.

### 11.1 — Outliers por IQR en `price`, filtrado a la misma ciudad del Punto 6 (París)

Cálculo manual: Q1, Q3, IQR = Q3 - Q1, límite inferior = Q1 - 1.5×IQR, límite superior = Q3 + 1.5×IQR.

### 11.2 — Basura no estadística (no es IQR, son valores inválidos o placeholders de la plataforma)

- `minimum_nights == 9999`: valor absurdo, listado prácticamente no reservable.
- `minimum_nights >= 365`: **no se trata como basura** — es plausible que sean alquileres de larga estancia.
- `maximum_nights`: dos placeholders distintos de "sin límite" (`1125` y `2147483647` = INT32_MAX), no son outliers reales de negocio.
- `accommodates == 0`: dato inválido, un alojamiento no puede alojar a 0 personas.
- `price == 0`: probablemente error de carga o placeholder, no un alojamiento gratuito real.

Nota para Fase 4/5: estos registros de basura (`accommodates == 0`, `price == 0`, `minimum_nights == 9999`, placeholders de `maximum_nights`) quedan disponibles en `df_clean` sin excluir. Queda a criterio de quien haga cada gráfico filtrarlos puntualmente si distorsionan una visualización específica, dejando la nota correspondiente en la interpretación de esa gráfica.

In [11]:
# --- 11.1 IQR sobre price en la ciudad filtrada (Punto 6) ---
price_city = df_clean.loc[df_clean["city"] == city_top, "price"]

Q1 = price_city.quantile(0.25)
Q3 = price_city.quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

print(f"IQR de price en {city_top}:")
print(f"  Q1            = {Q1}")
print(f"  Q3            = {Q3}")
print(f"  IQR           = {IQR}")
print(f"  limite_inf    = {limite_inf}")
print(f"  limite_sup    = {limite_sup}")

outliers_price = price_city[(price_city < limite_inf) | (price_city > limite_sup)]
print(f"\nRegistros fuera de rango: {len(outliers_price)} de {len(price_city)} ({len(outliers_price) / len(price_city) * 100:.2f}%)")
print("\nAlgunos ejemplos de outliers (price):")
print(outliers_price.sort_values(ascending=False).head(10))

# --- 11.2 Basura no estadistica (reporte, sin eliminar) ---
print("\n" + "=" * 60)
print("Basura no estadistica (reporte, NO se elimina):")
print("=" * 60)

min_nights_9999 = (df_clean["minimum_nights"] == 9999).sum()
min_nights_larga_estancia = (df_clean["minimum_nights"] >= 365).sum()
print(f"minimum_nights == 9999 (basura):              {min_nights_9999}")
print(f"minimum_nights >= 365 (larga estancia, valido): {min_nights_larga_estancia}")

max_nights_1125 = (df_clean["maximum_nights"] == 1125).sum()
max_nights_int32max = (df_clean["maximum_nights"] == 2147483647).sum()
print(f"\nmaximum_nights == 1125 (placeholder 'sin limite' #1):        {max_nights_1125}")
print(f"maximum_nights == 2147483647 / INT32_MAX (placeholder #2):   {max_nights_int32max}")

accom_zero = (df_clean["accommodates"] == 0).sum()
print(f"\naccommodates == 0 (invalido):  {accom_zero}")

price_zero = (df_clean["price"] == 0).sum()
print(f"price == 0 (invalido/placeholder): {price_zero}")


IQR de price en Paris:
  Q1            = 59.0
  Q3            = 120.0
  IQR           = 61.0
  limite_inf    = -32.5
  limite_sup    = 211.5

Registros fuera de rango: 5162 de 64657 (7.98%)

Algunos ejemplos de outliers (price):
36228     12000
60735     11599
227298    10250
28872      9631
45175      9280
45174      9280
60860      9280
60845      9280
225899     9280
64147      9280
Name: price, dtype: int64

Basura no estadistica (reporte, NO se elimina):
minimum_nights == 9999 (basura):              1
minimum_nights >= 365 (larga estancia, valido): 408

maximum_nights == 1125 (placeholder 'sin limite' #1):        157496
maximum_nights == 2147483647 / INT32_MAX (placeholder #2):   3

accommodates == 0 (invalido):  85
price == 0 (invalido/placeholder): 113


## Estado para Fase 3 (compañeros)

- **DataFrame limpio:** `df_clean`
- **Shape final:** ver salida de la celda siguiente (filas eliminadas: ~165 por metadata de host incompleta; 0 por duplicados).
- **Qué se limpió:**
  - Nulos estructurales (`district`, `host_response_time`, `host_response_rate`, `review_scores_*`) se dejaron como NaN o constante `"Sin datos"`, **no imputados con media/mediana/moda**.
  - `bedrooms` (10.52% nulos, aleatorio) imputado con la mediana.
  - Filas con metadata de host incompleta (`host_since`, `host_identity_verified`, `host_has_profile_pic`, `host_total_listings_count`, `host_is_superhost` nulos simultáneamente, ~165 filas) eliminadas.
  - `name`, `host_location`, `neighbourhood`: mojibake corregido (doble decodificación latin-1→utf-8).
  - Columnas booleanas `f`/`t` convertidas a `True`/`False`: `host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `instant_bookable`.
  - `host_since` convertido a `datetime` real.
  - 0 duplicados exactos y 0 duplicados lógicos por `listing_id` — no se eliminó nada por esta vía.
  - Outliers de `price` (por IQR) y basura no estadística (`minimum_nights == 9999`, placeholders de `maximum_nights`, `accommodates == 0`, `price == 0`) **reportados pero NO eliminados** — quedan disponibles para que Fase 3/4 decida cómo tratarlos.

- **Columnas flag creadas:**
  - `tiene_resenas` (bool): `True` si `review_scores_rating` no es nulo.
  - `tiene_respuesta_host` (bool): `True` si el host tiene historial de tiempo de respuesta.

- **Columnas candidatas para `groupby` / visualización:** `city`, `room_type`, `property_type`, `neighbourhood`, `host_is_superhost`, `instant_bookable`, `tiene_resenas`.

- **⚠️ Advertencia importante para comparaciones de precio:** `price` está en **moneda local de cada ciudad** (no hay tasas de cambio en el dataset). **Nunca comparar o promediar `price` entre ciudades distintas sin convertir a una moneda común primero** — cualquier `groupby('city')['price'].mean()` es válido *dentro* de cada ciudad, pero no comparable *entre* ciudades.


In [12]:
print("Shape final de df_clean:", df_clean.shape)
print("Filas originales:", df.shape[0])
print("Filas eliminadas en total:", df.shape[0] - df_clean.shape[0])
print("\nColumnas flag disponibles:", ["tiene_resenas", "tiene_respuesta_host"])
print("\ndf_clean.dtypes:")
print(df_clean.dtypes)


Shape final de df_clean: (279547, 35)
Filas originales: 279712
Filas eliminadas en total: 165

Columnas flag disponibles: ['tiene_resenas', 'tiene_respuesta_host']

df_clean.dtypes:
listing_id                              int64
name                                      str
host_id                                 int64
host_since                     datetime64[us]
host_location                             str
host_response_time                        str
host_response_rate                    float64
host_acceptance_rate                  float64
host_is_superhost                        bool
host_total_listings_count             float64
host_has_profile_pic                     bool
host_identity_verified                   bool
neighbourhood                             str
district                                  str
city                                      str
latitude                              float64
longitude                             float64
property_type                       

## Punto 12 — Categorías y conteos en columnas categóricas (Fase 3)

En el Punto 10 se usó `.unique()` sobre `room_type` y `city` solo para verificar consistencia de formato (mayúsculas, espacios), no para contar cuántos registros tiene cada categoría. Aquí se hace ese conteo sobre dos columnas categóricas distintas: `property_type` (muchas categorías, cola larga) y `host_is_superhost` (binaria).

In [13]:
# Categorias y conteo: cuantos registros tiene cada valor posible de la columna
print("property_type - categorias y conteo (value_counts):")
print(df_clean["property_type"].value_counts())

print("\nhost_is_superhost - categorias y conteo:")
print(df_clean["host_is_superhost"].value_counts())

property_type - categorias y conteo (value_counts):
property_type
Entire apartment                138897
Private room in apartment        47296
Private room in house            13289
Entire house                     13261
Entire condominium               11245
                                 ...  
Room in heritage hotel               1
Igloo                                1
Private room in cave                 1
Private room in holiday park         1
Tipi                                 1
Name: count, Length: 143, dtype: int64

host_is_superhost - categorias y conteo:
host_is_superhost
False    229294
True      50253
Name: count, dtype: int64


**Interpretación:** `property_type` tiene una categoría dominante — `Entire apartment` concentra ~49.7% de todos los listados (138,897 de 279,547) — seguida de lejos por `Private room in apartment` (~16.9%). El resto son categorías largas de cola (lofts, condominios, cuartos de hotel, etc.), cada una por debajo del 5%. En `host_is_superhost`, solo el ~18% de los hosts (50,253) tienen el estatus de superhost frente a un ~82% que no lo tiene (229,294) — es una condición selectiva, no la mitad de la plataforma.

## Punto 13 — Diccionario con métrica agregada por categoría filtrada

Se construye un diccionario donde la clave es cada categoría de `room_type` y el valor es el `review_scores_rating` promedio, calculado **solo sobre el subconjunto filtrado** de listados que son de reserva instantánea (`instant_bookable`) y que además tienen reseñas (`tiene_resenas`). No es el promedio general por `room_type`: es el promedio dentro de ese subconjunto específico (listados "fáciles de reservar y ya evaluados").

In [14]:
# Subconjunto filtrado: solo listados de reserva instantanea Y que ya tienen resenas
subset_reservable_evaluado = df_clean[df_clean["instant_bookable"] & df_clean["tiene_resenas"]]

# Diccionario categoria (room_type) -> metrica agregada (rating promedio), calculada
# UNICAMENTE dentro del subconjunto filtrado de arriba, no sobre todo df_clean
rating_promedio_por_room_type = {
    rt: round(subset_reservable_evaluado.loc[subset_reservable_evaluado["room_type"] == rt, "review_scores_rating"].mean(), 2)
    for rt in df_clean["room_type"].unique()
}

print(f"Tamano del subconjunto filtrado: {len(subset_reservable_evaluado)} de {len(df_clean)} listados")
print("\nRating promedio por room_type (dentro del subconjunto filtrado):")
print(rating_promedio_por_room_type)

Tamano del subconjunto filtrado: 74878 de 279547 listados

Rating promedio por room_type (dentro del subconjunto filtrado):
{'Entire place': np.float64(92.97), 'Private room': np.float64(91.53), 'Hotel room': np.float64(90.55), 'Shared room': np.float64(90.7)}


**Interpretación:** dentro de los listados de reserva instantánea y con reseñas, `Entire place` tiene el rating promedio más alto (92.97), seguido de `Private room` (91.53); `Hotel room` (90.55) y `Shared room` (90.70) quedan más abajo y prácticamente empatados entre sí. La diferencia es de menos de 2.5 puntos sobre una escala de 100 — no es una brecha grande, pero es consistente con la intuición de que tener el espacio completo para uno mismo se asocia con mejor experiencia reportada que compartirlo.

## Punto 14 — Preguntas de negocio con `.groupby()`

**Pregunta 1:** ¿cuál es el precio promedio por `room_type`, calculado dentro de cada ciudad? (se agrupa primero por `city` para respetar la advertencia de la Fase 2: `price` no es comparable entre ciudades, solo dentro de una misma ciudad).

**Pregunta 2:** ¿los superhosts obtienen, en promedio, mejor `review_scores_rating` que los hosts que no lo son?

In [15]:
# Pregunta 1: precio promedio por room_type, agrupando primero por city para no
# mezclar monedas distintas (price esta en moneda local de cada ciudad, ver Fase 2)
precio_promedio_por_ciudad_y_tipo = df_clean.groupby(["city", "room_type"])["price"].mean().round(2)
print("Pregunta 1 - precio promedio por room_type, dentro de cada ciudad:")
print(precio_promedio_por_ciudad_y_tipo)

# Pregunta 2: rating promedio comparando superhosts vs. hosts regulares
rating_por_superhost = df_clean.groupby("host_is_superhost")["review_scores_rating"].mean().round(2)
conteo_por_superhost = df_clean["host_is_superhost"].value_counts()
print("\nPregunta 2 - rating promedio: superhost vs. no superhost:")
print(rating_por_superhost)
print("\nConteo de listados por grupo:")
print(conteo_por_superhost)

Pregunta 1 - precio promedio por room_type, dentro de cada ciudad:
city            room_type   
Bangkok         Entire place    2155.50
                Hotel room      2090.75
                Private room    2037.35
                Shared room     1323.10
Cape Town       Entire place    2816.33
                Hotel room      2929.79
                Private room    1132.90
                Shared room      499.29
Hong Kong       Entire place    1016.65
                Hotel room       925.58
                Private room     570.02
                Shared room      635.55
Istanbul        Entire place     645.62
                Hotel room       644.08
                Private room     405.98
                Shared room      314.32
Mexico City     Entire place    1451.93
                Hotel room      1564.50
                Private room     796.14
                Shared room      743.24
New York        Entire place     191.46
                Hotel room       228.57
                Private 

**Interpretación:**

- **Pregunta 1:** en las 10 ciudades, `Entire place` es consistentemente más caro que `Private room` (esperable: se paga el espacio completo). La excepción llamativa es Río de Janeiro, donde `Shared room` promedia 1,458.99 — más caro que `Entire place` (823.02) — con un tamaño de grupo nada despreciable (n=611, no es un artefacto de muestra chica). Es la misma señal que ya se vio en el Punto 11: la media es sensible a outliers, así que ese promedio probablemente está inflado por un puñado de listados de precio extremo dentro de esa categoría, no por que "cuarto compartido" sea sistemáticamente más caro que el apartamento completo.
- **Pregunta 2:** los superhosts promedian 97.00 de rating frente a 92.26 de los que no lo son — una diferencia de ~4.7 puntos, la brecha más grande observada en este notebook entre dos grupos. Tiene sentido con el propio criterio de Airbnb para otorgar el estatus (exige buen historial de reseñas), pero también sugiere que `host_is_superhost` es una variable candidata fuerte para explicar `review_scores_rating` en un futuro modelo.

## Punto 15 — Top 5 / bottom 5 por `price` (dentro de una ciudad)

Igual que en el Punto 6 y el Punto 11, se ordena `price` filtrado a una sola ciudad (`city_top` = la ciudad con más registros) para no mezclar monedas distintas.

In [16]:
# Se ordena price filtrado a una sola ciudad (city_top, definida en el Punto 6)
# para que los 5 mas caros/baratos sean comparables entre si (misma moneda)
price_city_sorted = df_clean[df_clean["city"] == city_top].sort_values("price", ascending=False)

cols_interes = ["name", "room_type", "property_type", "accommodates", "price"]

print(f"Top 5 mas caros en {city_top}:")
display(price_city_sorted.head(5)[cols_interes])

# .tail(5) sobre el orden descendente = los 5 valores mas bajos
print(f"\nBottom 5 mas baratos en {city_top}:")
display(price_city_sorted.tail(5)[cols_interes])

Top 5 mas caros en Paris:


,name,room_type,property_type,accommodates,price
36228,Cute 1 bedroom flat in the center of Paris.,Entire place,Entire apartment,2,12000
60735,Amazing apartment 10P-St Marcel/Mouffetard MASQ,Entire place,Entire apartment,10,11599
227298,Maison Montespan,Entire place,Entire apartment,10,10250
28872,Charming 4P - Republique/ Temple,Entire place,Entire apartment,4,9631
225899,Amazing apart 4P - Canal StMartin MOBILITY LEASE,Entire place,Entire apartment,4,9280



Bottom 5 mas baratos en Paris:


,name,room_type,property_type,accommodates,price
207084,Hôtel Beaugrenelle Tour Eiffel,Hotel room,Room in boutique hotel,0,0
207086,25hours Hotel Terminus Nord,Hotel room,Room in hotel,0,0
208301,Le Cinq Codet,Hotel room,Room in hotel,2,0
208303,Le Derby Alma,Hotel room,Room in boutique hotel,0,0
203264,HOTEL DES NATIONS SAINT GERMAIN,Hotel room,Room in boutique hotel,0,0


**Interpretación:** el top 5 son apartamentos completos (`Entire apartment`) de precio muy alto (entre ~9,280 y 12,000), casi todos con capacidad para grupos grandes (4 a 10 huéspedes) — son propiedades premium/de lujo, no representan al listado típico. El bottom 5 tiene `price == 0` y `accommodates == 0` en todos los casos: no son alojamientos genuinamente gratuitos, son exactamente los registros de "basura no estadística" ya reportados en el Punto 11 (`price == 0`, `accommodates == 0`), la mayoría cuartos de hotel (`Hotel room`). Confirma que esos valores deberían tratarse como inválidos y no como el extremo inferior real del rango de precios.